# 🏷️ Label Semantics Analysis: name_only/L2/L3

**Experiment — Label Semantics Analysis using Generated Descriptions**

- **name_only:** Sadece label ismi ("world", "sports", ...)
- **L2 (LLM single):** Task-aware tek açıklama (12-25 kelime)
- **L3 (LLM multi):** Task-aware 3 açıklama (mean pooling)

**Total:** 189 experiments (9 datasets × 7 models × 3 modes)

**Config Location:** `src/label_descriptions/experiments/`

## 🚀 Setup

In [ ]:
# Clone from GitHub (RECOMMENDED)
!git clone https://github.com/EngindalgaMaku/zeroshot-text-classification-benchmark-.git
%cd zeroshot-text-classification-benchmark-
!pip install -q -r requirements.txt

# Verify config directory exists
!ls -la src/label_descriptions/experiments/ | head -20

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## ⚙️ Configuration

In [ ]:
# ⚙️ SETTINGS
SKIP_EXISTING = True  # Skip if results already exist
SHOW_PROGRESS = True  # Show progress after each group

# You can filter by dataset or model to run subset
FILTER_DATASET = None  # e.g., "ag_news" or None for all
FILTER_MODEL = None    # e.g., "all-mpnet-base-v2" or None for all

print(f"Skip existing: {SKIP_EXISTING}")
print(f"Show progress: {SHOW_PROGRESS}")
print(f"Filter dataset: {FILTER_DATASET or 'All'}")
print(f"Filter model: {FILTER_MODEL or 'All'}")

## 🔧 Helper Functions

In [ ]:
import json
import subprocess
from pathlib import Path
from IPython.display import display, HTML

def run_experiment(config_path, skip_existing=True):
    """Run single experiment."""
    cmd = ["python", "main.py", "--config", str(config_path)]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ FAILED: {config_path.name}")
        print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
        return None
    
    # Load metrics from new output location
    exp_name = config_path.stem
    metrics_file = Path("results/full_label_descriptions") / f"{exp_name}_metrics.json"
    
    if metrics_file.exists():
        with open(metrics_file) as f:
            return json.load(f)
    return None

def show_comparison(l1, l2, l3, dataset, model):
    """Show name_only/L2/L3 comparison."""
    html = f"<h3>{dataset} - {model}</h3><table border='1' style='border-collapse: collapse; font-family: monospace;'>"
    html += "<tr><th>Mode</th><th>Accuracy</th><th>Macro F1</th><th>Gain vs name_only</th></tr>"
    
    if l1:
        html += f"<tr><td><b>name_only</b></td><td>{l1['accuracy']:.4f}</td><td>{l1['macro_f1']:.4f}</td><td>-</td></tr>"
    else:
        html += "<tr><td><b>name_only</b></td><td colspan='3'>❌ FAILED</td></tr>"
    
    if l2:
        gain = f"+{(l2['accuracy'] - l1['accuracy']):.4f}" if l1 else "N/A"
        html += f"<tr><td><b>L2 (LLM 1)</b></td><td>{l2['accuracy']:.4f}</td><td>{l2['macro_f1']:.4f}</td><td>{gain}</td></tr>"
    else:
        html += "<tr><td><b>L2 (LLM 1)</b></td><td colspan='3'>❌ FAILED</td></tr>"
    
    if l3:
        gain = f"+{(l3['accuracy'] - l1['accuracy']):.4f}" if l1 else "N/A"
        html += f"<tr><td><b>L3 (LLM 3)</b></td><td>{l3['accuracy']:.4f}</td><td>{l3['macro_f1']:.4f}</td><td>{gain}</td></tr>"
    else:
        html += "<tr><td><b>L3 (LLM 3)</b></td><td colspan='3'>❌ FAILED</td></tr>"
    
    html += "</table><br>"
    display(HTML(html))

## 🏃 Run All Experiments

In [ ]:
# Group configs
config_dir = Path("src/label_descriptions/experiments")
configs_by_group = {}

for config_path in sorted(config_dir.glob("*.yaml")):
    # Parse filename: full_{dataset}_{model}_{mode}.yaml
    parts = config_path.stem.split('_')
    if len(parts) < 4 or parts[0] != "full":
        continue
    
    # Extract mode (last part) and model (second to last)
    mode = parts[-1]      # name_only, l2, or l3
    model = parts[-2]     # mpnet, e5, bge, etc.
    dataset = '_'.join(parts[1:-2])  # Everything between "full" and model
    
    # Apply filters
    if FILTER_DATASET and FILTER_DATASET not in dataset:
        continue
    if FILTER_MODEL and FILTER_MODEL not in model:
        continue
    
    base_name = f"{dataset}_{model}"
    if base_name not in configs_by_group:
        configs_by_group[base_name] = {}
    
    configs_by_group[base_name][mode] = config_path

print(f"Found {len(configs_by_group)} groups to run")
print(f"Total experiments: {len(configs_by_group) * 3}")

# Show first few groups as sample
for i, (name, modes) in enumerate(list(configs_by_group.items())[:3]):
    print(f"  {name}: {list(modes.keys())}")

In [ ]:
from tqdm.notebook import tqdm
import time

print("="*80)
print("STARTING LABEL SEMANTICS EXPERIMENTS")
print("="*80)

all_results = []
start_time = time.time()

for idx, (base_name, configs) in enumerate(tqdm(sorted(configs_by_group.items()), desc="Groups")):
    # Parse dataset and model from base_name
    parts = base_name.rsplit('_', 1)
    dataset = parts[0]
    model = parts[1] if len(parts) == 2 else "unknown"
    
    print(f"\n{'#'*80}")
    print(f"# {idx+1}/{len(configs_by_group)}: {dataset} - {model}")
    print(f"{'#'*80}")
    
    # Run name_only, L2, L3
    l1_metrics = run_experiment(configs.get('name_only'), SKIP_EXISTING) if 'name_only' in configs else None
    l2_metrics = run_experiment(configs.get('l2'), SKIP_EXISTING) if 'l2' in configs else None
    l3_metrics = run_experiment(configs.get('l3'), SKIP_EXISTING) if 'l3' in configs else None
    
    # Store results
    all_results.append({
        'dataset': dataset,
        'model': model,
        'l1_acc': l1_metrics['accuracy'] if l1_metrics else None,
        'l2_acc': l2_metrics['accuracy'] if l2_metrics else None,
        'l3_acc': l3_metrics['accuracy'] if l3_metrics else None,
        'l1_f1': l1_metrics['macro_f1'] if l1_metrics else None,
        'l2_f1': l2_metrics['macro_f1'] if l2_metrics else None,
        'l3_f1': l3_metrics['macro_f1'] if l3_metrics else None,
    })
    
    # Show comparison
    if SHOW_PROGRESS:
        show_comparison(l1_metrics, l2_metrics, l3_metrics, dataset, model)

elapsed = time.time() - start_time
print(f"\n{'='*80}")
print(f"ALL EXPERIMENTS COMPLETED!")
print(f"Time: {elapsed/60:.1f} minutes")
print(f"{'='*80}")

## 📊 Results Analysis

In [ ]:
import pandas as pd

# Create DataFrame
df_results = pd.DataFrame(all_results)

# Calculate gains
df_results['l2_gain'] = df_results['l2_acc'] - df_results['l1_acc']
df_results['l3_gain'] = df_results['l3_acc'] - df_results['l1_acc']
df_results['l3_vs_l2'] = df_results['l3_acc'] - df_results['l2_acc']

# Save
df_results.to_csv("results/label_semantics_l1_l2_l3.csv", index=False)
print("✅ Saved: results/label_semantics_l1_l2_l3.csv")

df_results.head(10)

In [ ]:
# Overall statistics
print("="*80)
print("OVERALL STATISTICS")
print("="*80)

print("\nMean Accuracy:")
print(f"L1 (name_only): {df_results['l1_acc'].mean():.4f}")
print(f"L2 (LLM 1):     {df_results['l2_acc'].mean():.4f}")
print(f"L3 (LLM 3):     {df_results['l3_acc'].mean():.4f}")

print("\nMean Gains:")
print(f"L2 vs L1: {df_results['l2_gain'].mean():+.4f} ({df_results['l2_gain'].mean()*100:+.2f}%)")
print(f"L3 vs L1: {df_results['l3_gain'].mean():+.4f} ({df_results['l3_gain'].mean()*100:+.2f}%)")
print(f"L3 vs L2: {df_results['l3_vs_l2'].mean():+.4f} ({df_results['l3_vs_l2'].mean()*100:+.2f}%)")

In [ ]:
# Per-dataset analysis
print("="*80)
print("PER-DATASET ANALYSIS")
print("="*80)

dataset_stats = df_results.groupby('dataset')[['l1_acc', 'l2_acc', 'l3_acc', 'l2_gain', 'l3_gain']].mean()
dataset_stats = dataset_stats.sort_values('l3_gain', ascending=False)
print(dataset_stats.to_string())

In [ ]:
# Per-model analysis
print("="*80)
print("PER-MODEL ANALYSIS")
print("="*80)

model_stats = df_results.groupby('model')[['l1_acc', 'l2_acc', 'l3_acc', 'l2_gain', 'l3_gain']].mean()
model_stats = model_stats.sort_values('l3_gain', ascending=False)
print(model_stats.to_string())

## 📈 Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Overall comparison
ax = axes[0, 0]
means = [df_results['l1_acc'].mean(), df_results['l2_acc'].mean(), df_results['l3_acc'].mean()]
ax.bar(['L1 (name)', 'L2 (LLM 1)', 'L3 (LLM 3)'], means, color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_ylabel('Accuracy')
ax.set_title('Overall Accuracy by Label Level')
ax.set_ylim([0, 1])
for i, v in enumerate(means):
    ax.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# 2. Gains distribution
ax = axes[0, 1]
ax.hist([df_results['l2_gain'], df_results['l3_gain']], bins=20, label=['L2 vs L1', 'L3 vs L1'], alpha=0.7)
ax.set_xlabel('Accuracy Gain')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Accuracy Gains')
ax.legend()
ax.axvline(0, color='red', linestyle='--', alpha=0.5)

# 3. Per-dataset comparison
ax = axes[1, 0]
dataset_means = df_results.groupby('dataset')[['l1_acc', 'l2_acc', 'l3_acc']].mean().sort_values('l3_acc')
dataset_means.plot(kind='barh', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_xlabel('Accuracy')
ax.set_title('Accuracy by Dataset and Label Level')
ax.legend(['L1', 'L2', 'L3'])

# 4. Heatmap of gains
ax = axes[1, 1]
pivot = df_results.pivot_table(index='dataset', columns='model', values='l3_gain')
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax, cbar_kws={'label': 'L3 vs L1 Gain'})
ax.set_title('L3 Accuracy Gain by Dataset and Model')

plt.tight_layout()
plt.savefig('results/label_semantics_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved: results/label_semantics_analysis.png")
plt.show()

## 💾 Download Results

In [ ]:
# Zip results
!zip -r label_semantics_results.zip results/full_label_descriptions/ results/label_semantics_l1_l2_l3.csv results/label_semantics_analysis.png

from google.colab import files
files.download('label_semantics_results.zip')

print("✅ Results downloaded!")

## ✅ Done!

**Results:**
- `results/full_label_descriptions/` - All metrics and predictions (189 experiments)
- `results/label_semantics_l1_l2_l3.csv` - Summary table
- `results/label_semantics_analysis.png` - Visualizations

**Configs Used:**
- `src/label_descriptions/experiments/` - 189 YAML configs
- Modes: `name_only`, `l2`, `l3`
- Models: mpnet, e5, bge, nomic, jina, instructor, qwen3